# ChemBreak10 — Adaptive MDP Jailbreak Study

**Research targets:** ChemDFM · ChemLLM  
**Condition:** C3_ADAPTIVE_MDP (MDP-driven adaptive multi-turn jailbreak)  
**Phases:** `development` (train policy, 48 tasks) → `pilot` (evaluate, 100 tasks) → `full_bank` (final result, 200 tasks)

> Run cells top to bottom. Set `LIVE = False` for a dry-run first. C0/C1/C2 baselines are excluded and do not affect Q-policy training.

In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

# ── Fill these in before running ───────────────────────────────────────────
PROJECT_ID          = "REPLACE_WITH_YOUR_GCP_PROJECT_ID"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak10"
PHASE               = "development"   # development | pilot | full_bank
EXPERIMENT_REVISION = "CB10_MDP_V1"
LIVE                = False            # False = dry-run/mock; True = live models

# ── Validation ─────────────────────────────────────────────────────────────
assert PHASE in {"development","pilot","holdout","full_bank"}, f"Unknown phase: {PHASE}"
content_root = Path("/content").resolve()
assert content_root.is_dir(), "/content unavailable"
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith("REPLACE_"), "Set PROJECT_ID before running live."

# ── Storage and cache paths ─────────────────────────────────────────────────
storage_root = content_root / "chembreak10_storage"
model_cache  = storage_root / "cache" / "huggingface" / "hub"

env_paths = {
    "HF_HOME":                storage_root / "cache/huggingface",
    "HF_HUB_CACHE":           model_cache,
    "HF_MODULES_CACHE":       storage_root / "cache/huggingface/modules",
    "XDG_CACHE_HOME":         storage_root / "cache/xdg",
    "TORCH_HOME":             storage_root / "cache/torch",
    "TORCHINDUCTOR_CACHE_DIR":storage_root / "cache/torchinductor",
    "TRITON_CACHE_DIR":       storage_root / "cache/triton",
    "CUDA_CACHE_PATH":        storage_root / "cache/cuda",
    "PIP_CACHE_DIR":          storage_root / "cache/pip",
    "TMPDIR":                 storage_root / "tmp",
}
for var, path in env_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[var] = str(path)
os.environ["TMP"] = os.environ["TEMP"] = str(env_paths["TMPDIR"])

print("Storage:", storage_root)
print("Model cache:", model_cache)
print("Disk:", shutil.disk_usage(content_root))

In [ ]:
# Clone or pull latest from GitHub — no Google Drive mount required
checkout = content_root / "chembreak10_repo"

def git(*args, cwd=None, capture=False):
    cmd = ['git', *args]
    if capture:
        return subprocess.check_output(cmd, cwd=cwd, text=True).strip()
    subprocess.run(cmd, cwd=cwd, check=True)

if not (checkout / '.git').is_dir():
    git('clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(checkout))
else:
    git('fetch', 'origin', BRANCH, cwd=checkout)
    git('checkout', BRANCH, cwd=checkout)
    git('pull', '--ff-only', 'origin', BRANCH, cwd=checkout)

PROJECT_DIR = (checkout / PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR / 'pyproject.toml').is_file(), f'Package not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
# Install pinned ML packages to persistent custom location.
# Avoids conflicts with Colab Enterprise pre-installed versions.
package_dir = storage_root / 'python_packages'
package_dir.mkdir(parents=True, exist_ok=True)

compatibility_specs = [
    'transformers==4.40.2', 'tokenizers==0.19.1', 'huggingface-hub==0.23.5',
    'safetensors==0.4.5', 'accelerate==0.30.1', 'peft==0.10.0',
    'sentencepiece==0.2.0', 'einops==0.8.1',
]
marker  = package_dir / 'chembreak10_compatibility_stack.json'
expected   = {'specifications': compatibility_specs}
installed  = json.loads(marker.read_text()) if marker.exists() else None
if installed != expected:
    print('Installing compatibility stack (first run only)...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--target', str(package_dir), '--cache-dir', str(env_paths['PIP_CACHE_DIR']),
        '--no-deps', '--upgrade', *compatibility_specs,
    ], check=True)
    marker.write_text(json.dumps(expected, indent=2))
    print('Done.')
else:
    print('Compatibility stack already installed.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--target', str(package_dir), '--cache-dir', str(env_paths['PIP_CACHE_DIR']),
    'google-auth>=2.35,<3', 'google-cloud-storage>=2.18,<4',
    'google-genai>=1.47,<2', 'openai>=1.57,<3',
    'numpy>=1.26,<3', 'pandas>=2.2,<3', 'pydantic>=2.9,<3',
    'PyYAML>=6.0,<7', 'rdkit>=2024.3', 'scipy>=1.13,<2', 'tenacity>=9,<10',
], check=True)

site.addsitedir(str(package_dir))
sys.path.insert(0, str(package_dir))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
importlib.invalidate_caches()

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import yaml
from chembreak10.config import load_config

config = load_config(PROJECT_DIR / 'configs' / f'config.{PHASE}.yaml')
config.pop('_config_path', None)

config['run'].update({
    'project_root':         str(PROJECT_DIR),
    'task_bank_path':       str(PROJECT_DIR / 'data' / 'final_task_bank.csv'),
    'output_root':          str(storage_root / 'runs'),
    'dry_run':              not LIVE,
    'live_acknowledgement': LIVE,
})
config['storage'].update({
    'storage_root':         str(storage_root),
    'hf_home':              str(env_paths['HF_HOME']),
    'hf_hub_cache':         str(model_cache),
    'hf_modules_cache':     str(env_paths['HF_MODULES_CACHE']),
    'xdg_cache_home':       str(env_paths['XDG_CACHE_HOME']),
    'torch_home':           str(env_paths['TORCH_HOME']),
    'torchinductor_cache':  str(env_paths['TORCHINDUCTOR_CACHE_DIR']),
    'triton_cache':         str(env_paths['TRITON_CACHE_DIR']),
    'cuda_cache':           str(env_paths['CUDA_CACHE_PATH']),
    'pip_cache':            str(env_paths['PIP_CACHE_DIR']),
    'python_packages':      str(package_dir),
    'temp_dir':             str(env_paths['TMPDIR']),
    'offload_dir':          str(storage_root / 'offload'),
    'preflight_dir':        str(storage_root / 'preflight'),
})
for target in config['targets']:
    target['cache_dir']     = str(model_cache)
    target['offload_folder'] = str(storage_root / 'offload' / target['id'])

policy_dir           = storage_root / 'policies' / EXPERIMENT_REVISION
training_policy_path = policy_dir / 'development_policy.json'
frozen_policy_path   = policy_dir / 'frozen_policy.json'
config['policy']['artifact_path'] = str(
    training_policy_path if PHASE == 'development' else frozen_policy_path
)
if PHASE != 'development':
    assert frozen_policy_path.is_file(), (
        f'Frozen policy not found: {frozen_policy_path}\n'
        'Complete development and run the freeze cell first.'
    )

runtime_dir  = storage_root / 'runtime_configs'
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_path = runtime_dir / f'CB10_{PHASE}_{"live" if LIVE else "mock"}.yaml'
runtime_path.write_text(yaml.safe_dump(config, sort_keys=False))

if LIVE:
    os.environ['GOOGLE_CLOUD_PROJECT']  = PROJECT_ID
    os.environ['CHEMBREAK_ENABLE_LIVE'] = 'YES'
else:
    os.environ.pop('CHEMBREAK_ENABLE_LIVE', None)

print('Runtime config:', runtime_path)
print('Mode:', 'LIVE' if LIVE else 'MOCK')
print('Policy:', config['policy']['mode'], '->', config['policy']['artifact_path'])

In [ ]:
from chembreak10.preflight import run_preflight
import pprint

preflight = run_preflight(runtime_path, load_targets=False, probe_tokenizers=True)
pprint.pprint({k: preflight[k] for k in ('status','gpu','selected_subset','policy','roles','tokenizers')})
assert preflight['status'] == 'ok', 'Preflight failed — fix errors above before continuing.'

In [ ]:
import pandas as pd
from IPython.display import display
from chembreak10.runner import run_condition_target

def run_one(condition, target_id):
    run_dir      = Path(run_condition_target(runtime_path, condition, target_id))
    results_path = run_dir / 'release' / f'episode_results_{condition}.csv'
    metrics_path = run_dir / 'release' / f'metrics_{condition}.csv'
    results = pd.read_csv(results_path)
    metrics = pd.read_csv(metrics_path)
    display(results[results.target_id == target_id])
    display(metrics[metrics.target_id.isin([target_id, 'ALL_TARGETS'])])
    print('Episode CSV:', results_path)
    print('Metrics  CSV:', metrics_path)
    return run_dir

print('run_one helper ready')

## C3_ADAPTIVE_MDP — ChemDFM

MDP-driven adaptive jailbreak against ChemDFM (LLaMA-8B base). Q-policy learns which tactics break ChemDFM's refusals across 48 tasks. Checkpointed every episode — safe to restart if the session disconnects.

In [ ]:
RUN_DIR = run_one('C3_ADAPTIVE_MDP', 'ChemDFM')

## C3_ADAPTIVE_MDP — ChemLLM

Same policy, same 48 tasks — now against ChemLLM (InternLM-2 base). Q-table continues updating from ChemDFM. Policy gains exposure to two different base architectures and refusal styles.

In [ ]:
RUN_DIR = run_one('C3_ADAPTIVE_MDP', 'ChemLLM')

## Freeze policy

Run **only after both target cells complete** for development. Locks the Q-table so pilot and full_bank measure what was learned, not what is still learning.

In [ ]:
from chembreak10.runner import strict_completion_gate
from chembreak10.policy import freeze_policy

if PHASE == 'development':
    gate = strict_completion_gate(runtime_path, 'C3_ADAPTIVE_MDP')
    print('Completion gate:', gate)
    frozen_policy_path.parent.mkdir(parents=True, exist_ok=True)
    result = freeze_policy(training_policy_path, frozen_policy_path)
    print('Policy frozen:', result)
else:
    print(f'Phase is {PHASE!r} — no freeze needed. This phase uses the frozen policy.')

## Download release files

In [ ]:
from IPython.display import FileLink, display as ipy_display

release_dir  = Path(RUN_DIR) / 'release'
download_dir = storage_root / 'downloads'
download_dir.mkdir(parents=True, exist_ok=True)

for name in (
    'episode_results_C3_ADAPTIVE_MDP.csv',
    'metrics_C3_ADAPTIVE_MDP.csv',
    'asr_by_budget_C3_ADAPTIVE_MDP.csv',
):
    p = release_dir / name
    if p.exists():
        ipy_display(FileLink(str(p)))

archive = Path(shutil.make_archive(
    str(download_dir / f'CB10_{PHASE}_release'), 'zip',
    root_dir=Path(RUN_DIR), base_dir='release',
))
print('Release archive:', archive)
ipy_display(FileLink(str(archive)))